# 🧠 ECLAT Skill Gap Analyzer — Google Colab Notebook
**Academic Project** | Association Rule Mining for IT Career Skill Recommendations

This notebook implements the full ECLAT (Equivalence Class Clustering and bottom-up Lattice Traversal) pipeline:
- Skill taxonomy & alias normalization
- Vertical TID-list construction & lattice traversal
- Association rule mining (Support, Confidence, Lift)
- 3-Tier fallback recommendation engine
- Dataset preprocessing & hybrid role classification
- Leave-One-Out Hit-Rate evaluation vs Naive Baseline
- Results visualization

> **Runtime:** ~2–3 minutes top-to-bottom in Google Colab (Python 3.10+)

## 📋 Table of Contents

| Section | Topic |
|---------|-------|
| [0](#section-0) | Setup & Install |
| [1](#section-1) | Skill Taxonomy & Alias Normalization |
| [2](#section-2) | ECLAT Algorithm Engine |
| [3](#section-3) | Interactive Demo — Synthetic Transactions |
| [4](#section-4) | Dataset Preprocessing Pipeline |
| [5](#section-5) | Offline Rule Precomputation |
| [6](#section-6) | Leave-One-Out Hit-Rate Evaluation |
| [7](#section-7) | Results Visualization |
| [8](#section-8) | Connect to Live FastAPI Server (Optional) |

## 🛠️ Section 0: Setup & Install <a id="section-0"></a>

Install required dependencies. `pandas`, `matplotlib`, and `collections`/`itertools` are
used throughout. `fastapi`, `uvicorn`, and `httpx` are only needed for Section 8.

In [ ]:
# Cell 0.1 — Install dependencies
!pip install pandas fastapi uvicorn httpx -q
print("✅ Dependencies installed.")

In [ ]:
# Cell 0.2 — Standard library imports used throughout the notebook
import json
import re
import random
from collections import defaultdict
from itertools import combinations
from typing import Dict, FrozenSet, List, Optional, Set, Tuple
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")
print("✅ Imports complete.")

## 🏷️ Section 1: Skill Taxonomy & Alias Normalization <a id="section-1"></a>

This section defines the canonical skill vocabulary and alias resolution layer.

**Key components:**
- `SKILL_TAXONOMY` — maps 70+ skills to 5 categories
- `ALIAS_MAP` — normalises raw tokens like `"reactjs"`, `"k8s"`, `"postgres"` to canonical names
- `_COMPILED_PATTERNS` — pre-compiled regex list sorted by length (longest-first) to prevent partial matches
- `extract_skills(text)` — scans free-form job description text and returns sorted canonical skills
- `get_category(skill)` — looks up which category a skill belongs to

In [ ]:
# ============================================================
# TAXONOMY — Skill categories, alias map, and extraction logic
# ============================================================

CAT_LANGUAGE     = "Programming Languages"
CAT_FRAMEWORK    = "Frameworks & Libraries"
CAT_DATABASE     = "Databases & Storage"
CAT_CLOUD_DEVOPS = "Cloud & DevOps"
CAT_DEV_TOOLS    = "Architecture & Developer Tools"

SKILL_TAXONOMY: Dict[str, str] = {
    "Python": CAT_LANGUAGE, "JavaScript": CAT_LANGUAGE, "TypeScript": CAT_LANGUAGE,
    "Java": CAT_LANGUAGE, "C++": CAT_LANGUAGE, "C#": CAT_LANGUAGE, "Go": CAT_LANGUAGE,
    "Rust": CAT_LANGUAGE, "PHP": CAT_LANGUAGE, "Ruby": CAT_LANGUAGE, "SQL": CAT_LANGUAGE,
    "HTML": CAT_LANGUAGE, "CSS": CAT_LANGUAGE, "Kotlin": CAT_LANGUAGE, "Swift": CAT_LANGUAGE,
    "Scala": CAT_LANGUAGE, "R": CAT_LANGUAGE, "Bash": CAT_LANGUAGE,
    "React": CAT_FRAMEWORK, "Angular": CAT_FRAMEWORK, "Vue.js": CAT_FRAMEWORK,
    "Next.js": CAT_FRAMEWORK, "Node.js": CAT_FRAMEWORK, "Express.js": CAT_FRAMEWORK,
    "Django": CAT_FRAMEWORK, "FastAPI": CAT_FRAMEWORK, "Flask": CAT_FRAMEWORK,
    "Spring Boot": CAT_FRAMEWORK, "ASP.NET": CAT_FRAMEWORK, "Laravel": CAT_FRAMEWORK,
    "Ruby on Rails": CAT_FRAMEWORK, "Tailwind CSS": CAT_FRAMEWORK, "Bootstrap": CAT_FRAMEWORK,
    "Redux": CAT_FRAMEWORK, "Pandas": CAT_FRAMEWORK, "NumPy": CAT_FRAMEWORK,
    "Scikit-learn": CAT_FRAMEWORK, "PyTorch": CAT_FRAMEWORK, "TensorFlow": CAT_FRAMEWORK,
    "PostgreSQL": CAT_DATABASE, "MySQL": CAT_DATABASE, "MongoDB": CAT_DATABASE,
    "Redis": CAT_DATABASE, "SQLite": CAT_DATABASE, "Oracle": CAT_DATABASE,
    "Microsoft SQL Server": CAT_DATABASE, "Cassandra": CAT_DATABASE,
    "Elasticsearch": CAT_DATABASE, "DynamoDB": CAT_DATABASE, "Snowflake": CAT_DATABASE,
    "Docker": CAT_CLOUD_DEVOPS, "Kubernetes": CAT_CLOUD_DEVOPS, "AWS": CAT_CLOUD_DEVOPS,
    "Azure": CAT_CLOUD_DEVOPS, "GCP": CAT_CLOUD_DEVOPS, "CI/CD": CAT_CLOUD_DEVOPS,
    "Git": CAT_CLOUD_DEVOPS, "GitHub Actions": CAT_CLOUD_DEVOPS, "Linux": CAT_CLOUD_DEVOPS,
    "Terraform": CAT_CLOUD_DEVOPS, "Ansible": CAT_CLOUD_DEVOPS, "Jenkins": CAT_CLOUD_DEVOPS,
    "Nginx": CAT_CLOUD_DEVOPS,
    "REST API": CAT_DEV_TOOLS, "GraphQL": CAT_DEV_TOOLS, "Microservices": CAT_DEV_TOOLS,
    "Kafka": CAT_DEV_TOOLS, "RabbitMQ": CAT_DEV_TOOLS, "Celery": CAT_DEV_TOOLS,
    "WebSockets": CAT_DEV_TOOLS, "Postman": CAT_DEV_TOOLS, "Power BI": CAT_DEV_TOOLS,
    "Tableau": CAT_DEV_TOOLS, "Excel": CAT_DEV_TOOLS, "Airflow": CAT_DEV_TOOLS,
    "Spark": CAT_DEV_TOOLS, "Hadoop": CAT_DEV_TOOLS,
}

ALIAS_MAP: Dict[str, str] = {
    "python": "Python", "python3": "Python", "javascript": "JavaScript", "js": "JavaScript",
    "typescript": "TypeScript", "ts": "TypeScript", "java": "Java", "c++": "C++",
    "cpp": "C++", "c#": "C#", "csharp": "C#", "golang": "Go", "go": "Go", "rust": "Rust",
    "php": "PHP", "ruby": "Ruby", "sql": "SQL", "html": "HTML", "html5": "HTML",
    "css": "CSS", "css3": "CSS", "bash": "Bash", "shell": "Bash",
    "react": "React", "react.js": "React", "reactjs": "React", "angular": "Angular",
    "angularjs": "Angular", "vue": "Vue.js", "vue.js": "Vue.js", "vuejs": "Vue.js",
    "next": "Next.js", "next.js": "Next.js", "nextjs": "Next.js", "node": "Node.js",
    "node.js": "Node.js", "nodejs": "Node.js", "express": "Express.js",
    "express.js": "Express.js", "expressjs": "Express.js", "django": "Django",
    "fastapi": "FastAPI", "flask": "Flask", "spring": "Spring Boot",
    "spring boot": "Spring Boot", "springboot": "Spring Boot", "asp.net": "ASP.NET",
    ".net": "ASP.NET", "dotnet": "ASP.NET", "laravel": "Laravel", "rails": "Ruby on Rails",
    "ruby on rails": "Ruby on Rails", "tailwind": "Tailwind CSS", "tailwindcss": "Tailwind CSS",
    "bootstrap": "Bootstrap", "redux": "Redux", "pandas": "Pandas", "numpy": "NumPy",
    "scikit-learn": "Scikit-learn", "sklearn": "Scikit-learn", "pytorch": "PyTorch",
    "tensorflow": "TensorFlow", "postgres": "PostgreSQL", "postgresql": "PostgreSQL",
    "mysql": "MySQL", "mongo": "MongoDB", "mongodb": "MongoDB", "redis": "Redis",
    "sqlite": "SQLite", "oracle": "Oracle", "sql server": "Microsoft SQL Server",
    "mssql": "Microsoft SQL Server", "cassandra": "Cassandra",
    "elasticsearch": "Elasticsearch", "dynamodb": "DynamoDB", "snowflake": "Snowflake",
    "docker": "Docker", "k8s": "Kubernetes", "kubernetes": "Kubernetes", "aws": "AWS",
    "amazon web services": "AWS", "azure": "Azure", "gcp": "GCP",
    "google cloud": "GCP", "google cloud platform": "GCP", "ci/cd": "CI/CD",
    "cicd": "CI/CD", "git": "Git", "github actions": "GitHub Actions", "linux": "Linux",
    "terraform": "Terraform", "ansible": "Ansible", "jenkins": "Jenkins", "nginx": "Nginx",
    "rest": "REST API", "rest api": "REST API", "restful": "REST API",
    "graphql": "GraphQL", "microservices": "Microservices", "kafka": "Kafka",
    "rabbitmq": "RabbitMQ", "celery": "Celery", "websocket": "WebSockets",
    "websockets": "WebSockets", "postman": "Postman", "power bi": "Power BI",
    "powerbi": "Power BI", "tableau": "Tableau", "excel": "Excel", "airflow": "Airflow",
    "spark": "Spark", "apache spark": "Spark", "hadoop": "Hadoop",
}

_COMPILED_PATTERNS = []
for _raw_token, _canonical in sorted(ALIAS_MAP.items(), key=lambda x: -len(x[0])):
    _escaped = re.escape(_raw_token)
    if _raw_token in ["c++", "c#", "ci/cd", ".net"]:
        _pattern = re.compile(rf"(?:^|[\s,;:(/]){_escaped}(?:$|[\s,;:!)/])", re.IGNORECASE)
    else:
        _pattern = re.compile(rf"\b{_escaped}\b", re.IGNORECASE)
    _COMPILED_PATTERNS.append((_pattern, _canonical))


def extract_skills(text: str) -> List[str]:
    """Scan free-form text and return sorted list of canonical skills found."""
    if not isinstance(text, str) or not text.strip():
        return []
    found: Set[str] = set()
    for pattern, canonical in _COMPILED_PATTERNS:
        if pattern.search(text):
            found.add(canonical)
    return sorted(found)


def get_category(skill: str) -> str:
    """Return the taxonomy category for a canonical skill name."""
    return SKILL_TAXONOMY.get(skill, "General Technology")


# Quick sanity test
_test = extract_skills("We need a python3 developer with reactjs, postgresql and k8s experience.")
print(f"✅ Taxonomy loaded: {len(SKILL_TAXONOMY)} skills, {len(ALIAS_MAP)} aliases")
print(f"   extract_skills test → {_test}")

## ⛏️ Section 2: ECLAT Algorithm Engine <a id="section-2"></a>

ECLAT (Equivalence Class Clustering and bottom-up Lattice Traversal) uses a **vertical data format**:
instead of scanning transactions row by row, it maintains a **TID-list** (Transaction ID list) for
each item — the set of transaction indices where that item appears.

**Core idea:**
- `support(A ∪ B) = |TID(A) ∩ TID(B)|` — computed by fast set intersection, **O(min(|TID_A|, |TID_B|))**
- Items are grouped into equivalence classes by shared prefix → recursive DFS lattice traversal
- No repeated dataset scans after the initial TID-list build pass

**Rule generation** filters by:
- **Support** = P(A ∪ B) — how often the itemset appears
- **Confidence** = P(B|A) = support(A ∪ B) / support(A) — rule reliability
- **Lift** = Confidence / P(B) — confidence relative to random chance (> 1.0 = positive correlation)

**3-Tier recommendation fallback:**
1. **Tier 1 — Exact Match:** Direct antecedent rule lookup for the user's exact skill set
2. **Tier 2 — Sub-Lattice Step-Down:** Best-matching subset antecedent (tries sizes 3→2→1)
3. **Tier 3 — Foundational Prior:** Most frequent skills not already possessed

In [ ]:
# ============================================================
# ECLATMiner — Full implementation (self-contained, no external imports)
# ============================================================

class ECLATMiner:
    """ECLAT frequent itemset miner and association rule engine."""

    def __init__(
        self,
        transactions: List[List[str]],
        min_support: float = 0.05,
        min_absolute_count: int = 3,
        max_itemset_size: int = 4,
    ):
        self.transactions = transactions
        self.n_transactions = len(transactions)
        self.min_support = min_support
        self.max_itemset_size = max_itemset_size
        self.min_support_count = max(min_absolute_count, int(min_support * self.n_transactions))

        # Inverted index: Skill -> Set of Transaction IDs
        self.tid_lists: Dict[str, Set[int]] = defaultdict(set)
        self._build_tid_lists()

        # Mined structures
        self.frequent_itemsets: Dict[FrozenSet[str], int] = {}
        self.rules: List[Dict] = []

    def _build_tid_lists(self):
        """Construct vertical TID-lists from transactions."""
        for tid, t in enumerate(self.transactions):
            for skill in set(t):
                self.tid_lists[skill].add(tid)

    def mine(self) -> Dict[FrozenSet[str], int]:
        """Mine frequent itemsets using vertical lattice traversal."""
        if self.n_transactions == 0:
            return {}

        # 1. Filter frequent 1-itemsets
        frequent_1_items = []
        for item, tids in self.tid_lists.items():
            if len(tids) >= self.min_support_count:
                fset = frozenset([item])
                self.frequent_itemsets[fset] = len(tids)
                frequent_1_items.append((item, tids))

        # Sort items by frequency (ascending) for efficient prefix-tree pruning
        frequent_1_items.sort(key=lambda x: len(x[1]))

        # 2. Run recursive equivalence class traversal
        self._eclat_recurse([], frequent_1_items)
        return self.frequent_itemsets

    def _eclat_recurse(
        self,
        prefix: List[str],
        items: List[Tuple[str, Set[int]]]
    ):
        """Recursive DFS lattice traversal using TID-list intersections."""
        for i in range(len(items)):
            item_i, tids_i = items[i]
            new_prefix = prefix + [item_i]
            if len(new_prefix) >= self.max_itemset_size:
                continue
            new_items = []

            for j in range(i + 1, len(items)):
                item_j, tids_j = items[j]

                # ECLAT Core: Vertical Set Intersection
                common_tids = tids_i & tids_j
                if len(common_tids) >= self.min_support_count:
                    candidate_itemset = frozenset(new_prefix + [item_j])
                    self.frequent_itemsets[candidate_itemset] = len(common_tids)
                    new_items.append((item_j, common_tids))

            if new_items:
                self._eclat_recurse(new_prefix, new_items)

    def generate_rules(
        self,
        min_confidence: float = 0.35,
        min_lift: float = 1.10
    ) -> List[Dict]:
        """Generate association rules (A -> B) from frequent itemsets."""
        if not self.frequent_itemsets:
            self.mine()

        rules = []
        n = self.n_transactions
        if n == 0:
            return []

        # Iterate over itemsets of size >= 2
        for itemset, itemset_count in self.frequent_itemsets.items():
            k = len(itemset)
            if k < 2:
                continue

            itemset_support = itemset_count / n

            # Single-consequent rules (X -> {y})
            for item in itemset:
                antecedent = itemset - {item}
                consequent = frozenset([item])

                ant_count = self.frequent_itemsets.get(antecedent)
                if not ant_count:
                    ant_tids = None
                    for s in antecedent:
                        if ant_tids is None:
                            ant_tids = set(self.tid_lists.get(s, set()))
                        else:
                            ant_tids &= self.tid_lists.get(s, set())
                    ant_count = len(ant_tids) if ant_tids else 0

                if ant_count == 0:
                    continue

                confidence = itemset_count / ant_count
                consequent_count = len(self.tid_lists.get(item, set()))
                consequent_support = consequent_count / n

                if consequent_support == 0:
                    continue

                lift = confidence / consequent_support

                if confidence >= min_confidence and lift >= min_lift:
                    rules.append({
                        "antecedent": sorted(antecedent),
                        "consequent": [item],
                        "support": round(itemset_support, 4),
                        "confidence": round(confidence, 4),
                        "lift": round(lift, 3),
                        "category": get_category(item),
                        "joint_count": itemset_count,
                        "antecedent_count": ant_count,
                    })

        # Sort rules by Lift * Confidence descending
        rules.sort(key=lambda r: -(r["lift"] * r["confidence"]))
        self.rules = rules

        # Hash map index for O(1) antecedent lookup
        self.rules_by_ant: Dict[FrozenSet[str], List[Dict]] = defaultdict(list)
        for r in rules:
            self.rules_by_ant[frozenset(r["antecedent"])].append(r)

        return rules

    def recommend(
        self,
        user_skills: List[str],
        top_k: int = 5
    ) -> Dict:
        """Execute 3-tier fallback recommendation engine."""
        user_set = set(user_skills)
        recommendations = []
        matched_subset = None
        tier_used = "Tier 3: Foundational Prior"
        sub_lattice_used = False
        active_rules = []

        if not hasattr(self, "rules_by_ant") or not self.rules_by_ant:
            self.rules_by_ant = defaultdict(list)
            for r in self.rules:
                self.rules_by_ant[frozenset(r["antecedent"])].append(r)

        # Tier 1: Exact Antecedent Match
        exact_matches = [
            r for r in self.rules_by_ant.get(frozenset(user_set), [])
            if r["consequent"][0] not in user_set
        ]

        if exact_matches:
            tier_used = "Tier 1: Exact Match"
            active_rules = exact_matches
            for r in exact_matches:
                rec_skill = r["consequent"][0]
                recommendations.append({
                    "skill": rec_skill,
                    "category": r["category"],
                    "confidence": r["confidence"],
                    "lift": r["lift"],
                    "support": r["support"],
                    "evidence_jobs_count": r["joint_count"],
                    "tier": tier_used,
                    "explanation": f"Frequently paired with your exact stack {r['antecedent']} (Lift: {r['lift']}x, Conf: {int(r['confidence']*100)}%).",
                })

        # Tier 2: Sub-Lattice Step-Down (bounded to antecedent sizes 3, 2, 1)
        if not recommendations and len(user_set) > 0:
            max_sub = min(3, len(user_set) - 1 if len(user_set) > 1 else 1)
            for sub_len in range(max_sub, 0, -1):
                best_subset_rules = []
                best_sub_support = -1

                for sub in combinations(sorted(user_set), sub_len):
                    sub_fset = frozenset(sub)
                    sub_rules = [
                        r for r in self.rules_by_ant.get(sub_fset, [])
                        if r["consequent"][0] not in user_set
                    ]
                    if sub_rules:
                        sub_support = max(r["support"] for r in sub_rules)
                        if sub_support > best_sub_support:
                            best_sub_support = sub_support
                            best_subset_rules = sub_rules
                            matched_subset = list(sub)

                if best_subset_rules:
                    sub_lattice_used = True
                    tier_used = "Tier 2: Sub-Lattice Complement"
                    active_rules = best_subset_rules
                    for r in best_subset_rules:
                        rec_skill = r["consequent"][0]
                        recommendations.append({
                            "skill": rec_skill,
                            "category": r["category"],
                            "confidence": r["confidence"],
                            "lift": r["lift"],
                            "support": r["support"],
                            "evidence_jobs_count": r["joint_count"],
                            "tier": tier_used,
                            "explanation": f"Recommended via strongest matched subset {matched_subset} (Lift: {r['lift']}x, Conf: {int(r['confidence']*100)}%).",
                        })
                    break

        # Tier 3: Unconditional 1-Itemset Baseline Prior
        if not recommendations:
            tier_used = "Tier 3: Foundational Prior"
            frequent_1 = [
                (list(k)[0], count)
                for k, count in self.frequent_itemsets.items()
                if len(k) == 1 and list(k)[0] not in user_set
            ]
            frequent_1.sort(key=lambda x: -x[1])
            for skill, count in frequent_1[:top_k]:
                supp = count / self.n_transactions if self.n_transactions > 0 else 0.0
                recommendations.append({
                    "skill": skill,
                    "category": get_category(skill),
                    "confidence": round(supp, 4),
                    "lift": 1.0,
                    "support": round(supp, 4),
                    "evidence_jobs_count": count,
                    "tier": tier_used,
                    "explanation": f"Foundational industry competency ({int(supp*100)}% of postings in this track require this).",
                })

        # Deduplicate recommendations preserving highest rank
        seen_skills = set()
        deduped = []
        for rec in recommendations:
            if rec["skill"] not in seen_skills:
                seen_skills.add(rec["skill"])
                deduped.append(rec)

        return {
            "tier_used": tier_used,
            "sub_lattice_used": sub_lattice_used,
            "matched_subset": matched_subset,
            "active_rules": active_rules[:top_k * 2],
            "recommendations": deduped[:top_k],
        }


print("✅ ECLATMiner class defined successfully.")

## 🧪 Section 3: Interactive Demo — Synthetic Transactions <a id="section-3"></a>

We use **50 synthetic IT job posting transactions** across 3 role tracks (Backend Developer,
Frontend Developer, Data Analyst) to demonstrate the full pipeline without needing any external dataset.

Each transaction is a list of canonical skills from Section 1's taxonomy.

In [ ]:
# ============================================================
# SYNTHETIC DEMO DATA — 50 IT job transactions across 3 roles
# ============================================================

DEMO_TRANSACTIONS = {
    "Backend Developer": {
        "all": [
            {"skills": ["Python", "Django", "PostgreSQL", "Docker", "Git", "REST API"]},
            {"skills": ["Python", "FastAPI", "PostgreSQL", "Redis", "Docker", "Linux"]},
            {"skills": ["Python", "Flask", "MySQL", "Git", "REST API", "Linux"]},
            {"skills": ["Java", "Spring Boot", "PostgreSQL", "Docker", "Kubernetes", "REST API"]},
            {"skills": ["Java", "Spring Boot", "MySQL", "Redis", "Docker", "Microservices"]},
            {"skills": ["Python", "Django", "Redis", "Celery", "Docker", "Microservices"]},
            {"skills": ["Go", "PostgreSQL", "Docker", "Kubernetes", "REST API", "Linux"]},
            {"skills": ["Python", "FastAPI", "MongoDB", "Docker", "Git", "REST API"]},
            {"skills": ["Python", "Django", "PostgreSQL", "Redis", "Celery", "Linux"]},
            {"skills": ["Java", "Spring Boot", "Oracle", "Docker", "Jenkins", "Microservices"]},
            {"skills": ["Python", "Flask", "PostgreSQL", "Docker", "CI/CD", "Git"]},
            {"skills": ["Python", "FastAPI", "PostgreSQL", "Docker", "Linux", "Microservices"]},
            {"skills": ["Go", "Docker", "Kubernetes", "PostgreSQL", "Linux", "CI/CD"]},
            {"skills": ["Python", "Django", "MySQL", "Redis", "Git", "Linux"]},
            {"skills": ["Java", "Spring Boot", "PostgreSQL", "Kafka", "Docker", "REST API"]},
            {"skills": ["Python", "FastAPI", "Redis", "Celery", "Docker", "Kubernetes"]},
            {"skills": ["Python", "Flask", "SQLite", "Git", "REST API", "Linux"]},
            {"skills": ["Java", "Spring Boot", "MongoDB", "Docker", "CI/CD", "Microservices"]},
            {"skills": ["Python", "Django", "PostgreSQL", "Git", "REST API", "Docker"]},
            {"skills": ["Go", "PostgreSQL", "Redis", "Docker", "Linux", "REST API"]},
        ],
        "entry": [
            {"skills": ["Python", "Django", "PostgreSQL", "Git", "REST API"]},
            {"skills": ["Python", "Flask", "MySQL", "Git", "Linux"]},
            {"skills": ["Java", "Spring Boot", "MySQL", "Git", "REST API"]},
            {"skills": ["Python", "FastAPI", "PostgreSQL", "Docker", "Git"]},
            {"skills": ["Python", "Django", "SQLite", "Git", "Linux"]},
        ]
    },
    "Frontend Developer": {
        "all": [
            {"skills": ["JavaScript", "React", "CSS", "HTML", "Git", "REST API"]},
            {"skills": ["TypeScript", "React", "Redux", "CSS", "HTML", "Git"]},
            {"skills": ["JavaScript", "Vue.js", "CSS", "HTML", "Git", "REST API"]},
            {"skills": ["TypeScript", "Angular", "CSS", "HTML", "Git", "Docker"]},
            {"skills": ["JavaScript", "Next.js", "React", "Tailwind CSS", "HTML", "Git"]},
            {"skills": ["TypeScript", "React", "Next.js", "CSS", "HTML", "CI/CD"]},
            {"skills": ["JavaScript", "React", "Redux", "REST API", "Git", "HTML"]},
            {"skills": ["TypeScript", "Vue.js", "CSS", "HTML", "Node.js", "Git"]},
            {"skills": ["JavaScript", "Angular", "HTML", "CSS", "Git", "REST API"]},
            {"skills": ["React", "TypeScript", "Redux", "HTML", "CSS", "Docker"]},
        ],
        "entry": [
            {"skills": ["JavaScript", "React", "HTML", "CSS", "Git"]},
            {"skills": ["JavaScript", "Vue.js", "HTML", "CSS", "Git"]},
            {"skills": ["TypeScript", "React", "HTML", "CSS", "Git"]},
        ]
    },
    "Data Analyst": {
        "all": [
            {"skills": ["Python", "SQL", "Pandas", "NumPy", "Excel", "Tableau"]},
            {"skills": ["Python", "SQL", "Pandas", "Scikit-learn", "Git", "Power BI"]},
            {"skills": ["SQL", "Excel", "Tableau", "Power BI", "Python", "Pandas"]},
            {"skills": ["Python", "SQL", "Pandas", "NumPy", "Airflow", "Snowflake"]},
            {"skills": ["SQL", "Python", "Tableau", "Excel", "PostgreSQL", "Git"]},
            {"skills": ["Python", "SQL", "Pandas", "Spark", "Airflow", "AWS"]},
            {"skills": ["SQL", "Excel", "Power BI", "Python", "PostgreSQL", "Tableau"]},
            {"skills": ["Python", "SQL", "Pandas", "NumPy", "Scikit-learn", "Tableau"]},
            {"skills": ["SQL", "Python", "Excel", "Airflow", "Snowflake", "Power BI"]},
            {"skills": ["Python", "SQL", "Pandas", "Hadoop", "Spark", "Linux"]},
        ],
        "entry": [
            {"skills": ["SQL", "Excel", "Python", "Pandas", "Tableau"]},
            {"skills": ["SQL", "Python", "Excel", "Power BI", "Git"]},
            {"skills": ["SQL", "Excel", "Tableau", "Python", "Git"]},
        ]
    }
}

# Total transaction count
total = sum(len(v[m]) for v in DEMO_TRANSACTIONS.values() for m in v)
print(f"✅ Synthetic dataset loaded: {total} transactions across {len(DEMO_TRANSACTIONS)} roles")
for role, modes in DEMO_TRANSACTIONS.items():
    print(f"   {role:25s} → {len(modes['all'])} general  |  {len(modes['entry'])} entry-level")

In [ ]:
# ============================================================
# MINE frequent itemsets on Backend Developer "all" track
# ============================================================

be_transactions = [r["skills"] for r in DEMO_TRANSACTIONS["Backend Developer"]["all"]]

miner_demo = ECLATMiner(
    be_transactions,
    min_support=0.2,
    min_absolute_count=2,
    max_itemset_size=3,
)
frequent_itemsets_demo = miner_demo.mine()
rules_demo = miner_demo.generate_rules(min_confidence=0.3, min_lift=1.1)

print(f"✅ Mining complete on Backend Developer (all) — {len(be_transactions)} transactions")
print(f"   Frequent itemsets found : {len(frequent_itemsets_demo)}")
print(f"   Association rules derived: {len(rules_demo)}")

In [ ]:
# ============================================================
# Print frequent itemsets sorted by count (top 20)
# ============================================================

print("=" * 60)
print("TOP FREQUENT ITEMSETS (Backend Developer — All Levels)")
print("=" * 60)

sorted_itemsets = sorted(frequent_itemsets_demo.items(), key=lambda x: -x[1])
for i, (itemset, count) in enumerate(sorted_itemsets[:20], 1):
    support_pct = count / len(be_transactions) * 100
    items_str = ", ".join(sorted(itemset))
    print(f"  {i:2d}. [{items_str}]  →  count={count}  ({support_pct:.0f}%)")

In [ ]:
# ============================================================
# Print top 10 association rules
# ============================================================

print("=" * 70)
print("TOP 10 ASSOCIATION RULES (Backend Developer — All Levels)")
print("=" * 70)
print(f"{'#':>3}  {'Antecedent':<35} → {'Consequent':<18} Conf   Lift   Supp")
print("-" * 70)

for i, r in enumerate(rules_demo[:10], 1):
    ant_str = ", ".join(r["antecedent"])
    cons_str = r["consequent"][0]
    print(f"  {i:2d}. {ant_str:<35} → {cons_str:<18}  {r['confidence']:.2f}  {r['lift']:.3f}  {r['support']:.3f}")

In [ ]:
# ============================================================
# Run 3-tier recommendations for two user profiles
# ============================================================

print("\n" + "=" * 70)
print("RECOMMENDATION DEMO")
print("=" * 70)

for user_skills in [["Python", "SQL"], ["React", "CSS"]]:
    result = miner_demo.recommend(user_skills, top_k=5)
    print(f"\n👤 User skills: {user_skills}")
    print(f"   Tier used   : {result['tier_used']}")
    if result["matched_subset"]:
        print(f"   Matched subset: {result['matched_subset']}")
    print(f"   Recommendations:")
    for rec in result["recommendations"]:
        print(f"     • {rec['skill']:<20} [{rec['category']}]  Conf={rec['confidence']:.2f}  Lift={rec['lift']:.2f}")
        print(f"       {rec['explanation']}")

## 📦 Section 4: Dataset Preprocessing Pipeline <a id="section-4"></a>

This section implements the **hybrid role classifier** that processes the 516 MB LinkedIn
`linkedin_job_postings.csv` dataset into clean skill transaction databases.

**Pipeline overview:**
1. Stream CSV in 25,000-row chunks (memory efficient — stays well under Colab's 12 GB RAM)
2. Skip postings with < 40 chars description or < 3 extractable skills
3. `classify_role()` — hybrid title+stack classifier assigns one of 3 tracks
4. `is_entry_level()` — multi-signal detector (metadata field + title regex + description regex)
5. Export `clean_transactions.json` and `clean_summary.json` to `data/processed/`

> **Note:** You need the actual `linkedin_job_postings.csv` to run `build_clean_dataset()`.
> The demo in Sections 3, 5, 6, and 7 uses the synthetic data defined above.

In [ ]:
# ============================================================
# HYBRID ROLE CLASSIFIER — classify_role() and is_entry_level()
# ============================================================
import re as _re

RE_BACKEND_TITLE = _re.compile(
    r"\b(backend|back-end|api engineer|python developer|java developer|golang developer|c# developer|node developer|django developer|spring developer)\b",
    _re.IGNORECASE,
)
RE_FRONTEND_TITLE = _re.compile(
    r"\b(frontend|front-end|ui engineer|ui developer|react developer|vue developer|angular developer|web developer|javascript developer)\b",
    _re.IGNORECASE,
)
RE_DATA_TITLE = _re.compile(
    r"\b(data analyst|business intelligence analyst|bi analyst|sql analyst|analytics engineer|data specialist|reporting analyst)\b",
    _re.IGNORECASE,
)
RE_GENERIC_SWE_TITLE = _re.compile(
    r"\b(software engineer|software developer|application developer|programmer)\b",
    _re.IGNORECASE,
)
RE_JUNIOR_TITLE = _re.compile(
    r"\b(junior|jr|associate|intern|internship|graduate|entry level|entry-level|trainee|apprentice|l1)\b",
    _re.IGNORECASE,
)
RE_JUNIOR_DESC = _re.compile(
    r"\b(0-2 years|0 to 2 years|1-2 years|1 to 2 years|recent graduate|no experience required|entry level|entry-level)\b",
    _re.IGNORECASE,
)


def classify_role(title: str, skills: List[str]) -> Optional[str]:
    """Hybrid Classifier: Combines job title with extracted stack to assign track."""
    if not isinstance(title, str):
        return None

    t_clean = title.strip()
    skills_set = set(skills)

    # 1. Direct explicit title matches
    if RE_BACKEND_TITLE.search(t_clean) and not RE_FRONTEND_TITLE.search(t_clean):
        return "Backend Developer"
    if RE_FRONTEND_TITLE.search(t_clean) and not RE_BACKEND_TITLE.search(t_clean):
        return "Frontend Developer"
    if RE_DATA_TITLE.search(t_clean):
        return "Data Analyst"

    # 2. Hybrid Disambiguation for Generic SWE/Dev titles
    if RE_GENERIC_SWE_TITLE.search(t_clean):
        backend_markers = {"Django", "FastAPI", "Flask", "Spring Boot", "ASP.NET",
                           "PostgreSQL", "MySQL", "Redis", "Microservices", "REST API", "Docker", "Go", "Java"}
        frontend_markers = {"React", "Vue.js", "Angular", "Next.js", "CSS", "HTML",
                            "Tailwind CSS", "Redux", "TypeScript"}
        data_markers = {"Pandas", "NumPy", "Tableau", "Power BI", "SQL", "Excel",
                        "Airflow", "Snowflake"}

        be_score = len(skills_set & backend_markers)
        fe_score = len(skills_set & frontend_markers)
        da_score = len(skills_set & data_markers)

        if be_score >= 2 and be_score > fe_score and be_score > da_score:
            return "Backend Developer"
        if fe_score >= 2 and fe_score > be_score:
            return "Frontend Developer"
        if da_score >= 2 and da_score > be_score:
            return "Data Analyst"

    return None


def is_entry_level(exp_level: Optional[str], title: str, description: str) -> bool:
    """Multi-signal check to detect Entry-Level / Junior postings."""
    if isinstance(exp_level, str):
        if exp_level.strip() in ["Entry level", "Associate", "Internship"]:
            return True
    if isinstance(title, str) and RE_JUNIOR_TITLE.search(title):
        return True
    if isinstance(description, str) and RE_JUNIOR_DESC.search(description):
        return True
    return False


print("✅ classify_role() and is_entry_level() defined")

# Quick tests
print(f"   classify_role('Senior Backend Developer', ['Python','Django','PostgreSQL']) → "
      f"{classify_role('Senior Backend Developer', ['Python','Django','PostgreSQL'])}")
print(f"   classify_role('Software Engineer', ['React','TypeScript','CSS','HTML']) → "
      f"{classify_role('Software Engineer', ['React','TypeScript','CSS','HTML'])}")
print(f"   is_entry_level('Entry level', 'Junior Python Dev', '') → "
      f"{is_entry_level('Entry level', 'Junior Python Dev', '')}")

In [ ]:
# ============================================================
# build_clean_dataset() — adapted to accept csv_path parameter
# ============================================================
import pandas as pd
from pathlib import Path

def build_clean_dataset(csv_path: str, limit_rows: Optional[int] = None, output_dir: str = ".") -> Dict:
    """Stream linkedin_job_postings.csv, extract skills, categorize roles, and save clean transactions.

    Args:
        csv_path: Path to your linkedin_job_postings.csv file (the 516 MB LinkedIn dataset)
        limit_rows: Optional row limit for quick testing (e.g., 50000)
        output_dir: Directory to write clean_transactions.json and clean_summary.json
    """
    postings_path = Path(csv_path)
    if not postings_path.exists():
        raise FileNotFoundError(f"Missing {postings_path}. Upload it in the next cell.")

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    print(f"Streaming and processing {postings_path.name}...")

    records_by_role_and_exp = {
        "Backend Developer":  {"entry": [], "all": []},
        "Frontend Developer": {"entry": [], "all": []},
        "Data Analyst":       {"entry": [], "all": []},
    }

    chunk_size = 25000
    total_scanned = 0
    total_retained = 0

    cols = ["job_id", "title", "formatted_experience_level", "description"]

    for chunk in pd.read_csv(postings_path, usecols=cols, chunksize=chunk_size):
        for _, row in chunk.iterrows():
            total_scanned += 1
            if limit_rows and total_scanned > limit_rows:
                break

            desc = str(row["description"]) if pd.notnull(row["description"]) else ""
            if len(desc) < 40:
                continue

            skills = extract_skills(desc)
            if len(skills) < 3:
                continue

            title = str(row["title"]) if pd.notnull(row["title"]) else ""
            role = classify_role(title, skills)
            if not role:
                continue

            exp_level = str(row["formatted_experience_level"]) if pd.notnull(row["formatted_experience_level"]) else None
            entry_flag = is_entry_level(exp_level, title, desc)

            item = {
                "job_id": int(row["job_id"]) if pd.notnull(row["job_id"]) else 0,
                "title": title,
                "skills": skills,
                "is_entry": entry_flag,
            }

            records_by_role_and_exp[role]["all"].append(item)
            if entry_flag:
                records_by_role_and_exp[role]["entry"].append(item)
            total_retained += 1

        print(f"  Scanned {total_scanned:,} postings... Retained {total_retained:,} clean IT transactions.", end="\r")
        if limit_rows and total_scanned > limit_rows:
            break

    print()
    # Export
    output_json = output_path / "clean_transactions.json"
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(records_by_role_and_exp, f, indent=2)

    summary = {}
    print("\n=== Dataset Preprocessing Complete ===")
    for role, modes in records_by_role_and_exp.items():
        all_count = len(modes["all"])
        entry_count = len(modes["entry"])
        summary[role] = {"all": all_count, "entry": entry_count}
        print(f"  {role:22s} | Entry-Level: {entry_count:5,d} | Total: {all_count:5,d}")

    summary_file = output_path / "clean_summary.json"
    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"\nArtifacts saved to: {output_path}/")
    return summary


print("✅ build_clean_dataset() defined (accepts csv_path parameter)")

### ⬆️ Optional: Upload Your `linkedin_job_postings.csv`

If you have the 516 MB LinkedIn Job Postings dataset, upload it here.
Otherwise, **skip this cell** — the demo uses synthetic data from Section 3.

In [ ]:
# OPTIONAL — Upload linkedin_job_postings.csv and run the full preprocessing pipeline
# Skip this cell if you don't have the dataset

RUN_UPLOAD = False  # Set to True to enable

if RUN_UPLOAD:
    try:
        from google.colab import files
        print("Please upload your linkedin_job_postings.csv file (516 MB)...")
        uploaded = files.upload()
        if uploaded:
            csv_filename = list(uploaded.keys())[0]
            print(f"\nUploaded: {csv_filename}")
            summary = build_clean_dataset(
                csv_path=csv_filename,
                limit_rows=None,  # Set e.g. 100000 for a quick test run
                output_dir="./data/processed"
            )
        else:
            print("No file uploaded.")
    except ImportError:
        print("Not running in Google Colab — use build_clean_dataset(csv_path='path/to/linkedin_job_postings.csv') directly.")
else:
    print("ℹ️  Upload skipped. Sections 5–7 will use synthetic DEMO_TRANSACTIONS.")

## 🔁 Section 5: Offline Rule Precomputation <a id="section-5"></a>

In production, the FastAPI backend loads a **precomputed rule base** from `data/processed/rules.json`
to stay within Render's 512 MB free tier memory limit. This section implements the offline miner
that generates those serialized rules.

The adapted `run_eclat_offline()` function accepts a `transactions_dict` directly (instead of
reading from a file), so it works with both the synthetic demo data and real preprocessed data.

In [ ]:
# ============================================================
# OFFLINE RULE MINER — adapted to accept transactions dict directly
# ============================================================

OFFLINE_MIN_SUPPORT = 0.05
OFFLINE_MIN_CONFIDENCE = 0.35
OFFLINE_MIN_LIFT = 1.15
OFFLINE_MIN_ABSOLUTE_COUNT = 3
OFFLINE_MAX_RULES_PER_MODE = 2000


def run_eclat_offline(transactions_dict: Dict, save_path: Optional[str] = None) -> Dict:
    """Mine rules across all roles and modes from a transactions dictionary.

    Args:
        transactions_dict: Dict like DEMO_TRANSACTIONS — {role: {mode: [{skills:[...]}]}}
        save_path: Optional file path to serialize the rule base as JSON

    Returns:
        precomputed dict with structure {role: {mode: {rules, top_1_itemsets, total_transactions}}}
    """
    precomputed = {}

    for role, modes in transactions_dict.items():
        precomputed[role] = {}
        for mode, records in modes.items():
            transactions = [r["skills"] for r in records]
            n = len(transactions)
            print(f"Mining [{role}] — Mode: [{mode}]  ({n} transactions)...", end=" ")

            if n < 5:
                print(f"⚠️  Insufficient ({n}) — skipped.")
                precomputed[role][mode] = {
                    "total_transactions": n,
                    "top_1_itemsets": [],
                    "rules": []
                }
                continue

            minsup  = OFFLINE_MIN_SUPPORT
            minconf = OFFLINE_MIN_CONFIDENCE
            minlift = OFFLINE_MIN_LIFT

            miner = ECLATMiner(
                transactions,
                min_support=minsup,
                min_absolute_count=OFFLINE_MIN_ABSOLUTE_COUNT,
                max_itemset_size=4,
            )
            miner.mine()
            rules = miner.generate_rules(min_confidence=minconf, min_lift=minlift)
            rules = rules[:OFFLINE_MAX_RULES_PER_MODE]

            # Top 1-itemsets for baseline priors
            top_1 = []
            for itemset, count in miner.frequent_itemsets.items():
                if len(itemset) == 1:
                    skill = list(itemset)[0]
                    top_1.append({
                        "skill": skill,
                        "category": get_category(skill),
                        "count": count,
                        "support": round(count / n, 4),
                    })
            top_1.sort(key=lambda x: -x["count"])

            precomputed[role][mode] = {
                "total_transactions": n,
                "top_1_itemsets": top_1[:20],
                "rules": rules,
            }

            print(f"✅  {len(miner.frequent_itemsets)} itemsets, {len(rules)} rules"
                  + (f"  |  Top rule: {rules[0]['antecedent']} → {rules[0]['consequent']}" if rules else ""))

    if save_path:
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(precomputed, f, indent=2)
        import os
        size_kb = os.path.getsize(save_path) / 1024
        print(f"\nRule base serialized to: {save_path}  ({size_kb:.1f} KB)")

    return precomputed


print("✅ run_eclat_offline() defined")

In [ ]:
# ============================================================
# Run offline mining on the synthetic demo data
# ============================================================

print("Running offline ECLAT on DEMO_TRANSACTIONS...\n")
offline_rules = run_eclat_offline(DEMO_TRANSACTIONS, save_path=None)

print("\n=== Offline Mining Summary ===")
for role in offline_rules:
    for mode in offline_rules[role]:
        d = offline_rules[role][mode]
        print(f"  [{role} — {mode}]  Transactions: {d['total_transactions']:3d}  |  Rules: {len(d['rules']):3d}  |  Top-1 items: {len(d['top_1_itemsets'])}")

## 📊 Section 6: Leave-One-Out Hit-Rate Evaluation <a id="section-6"></a>

**Evaluation methodology — Masked Skill Recovery:**
1. Split transactions 80% training / 20% test (seeded for reproducibility)
2. For each test transaction: randomly mask 1 skill → present the remaining skills to both ECLAT and the Naive Baseline
3. Measure whether the masked skill appears in Top-3 / Top-5 recommendations
4. **Non-trivial stratification:** separately evaluate skills with marginal frequency < 30% to prove ECLAT captures context, not just common words

**Metrics:**
- `HR@K` = Hit-Rate at K: fraction of test cases where masked skill is in top-K recommendations  
- `Non-Trivial HR@5` = HR@5 restricted to long-tail (< 30% frequency) masked skills  
- `Lift over Baseline %` = relative improvement of ECLAT vs Naive Popularity Baseline

> **Note:** The synthetic demo dataset has only 20 backend / 10 frontend / 10 data-analyst general transactions,
> which is below the 20-transaction minimum for statistical significance. The evaluation will note when a segment
> is too small. For meaningful results, use the full LinkedIn dataset.

In [ ]:
# ============================================================
# EVALUATION ENGINE — adapted to accept transactions dict directly
# ============================================================

EVAL_DEFAULT_MIN_SUPPORT    = 0.05
EVAL_DEFAULT_MIN_CONFIDENCE = 0.35
EVAL_DEFAULT_MIN_LIFT       = 1.15
EVAL_MIN_ABSOLUTE_COUNT     = 3


def evaluate_role_mode(
    records: List[Dict],
    role: str,
    mode: str,
    seed: int = 42
) -> Dict:
    """Leave-One-Out cross validation on a single role + experience segment."""
    random.seed(seed)

    valid_transactions = [r["skills"] for r in records if len(r["skills"]) >= 3]
    n_total = len(valid_transactions)

    if n_total < 20:
        print(f"  ⚠️  [{role} — {mode}] Only {n_total} transactions — below 20 minimum; returning zeros.")
        return {
            "role": role, "experience_mode": mode,
            "test_transactions_count": 0,
            "eclat_hr_at_3": 0.0, "baseline_hr_at_3": 0.0,
            "eclat_hr_at_5": 0.0, "baseline_hr_at_5": 0.0,
            "non_trivial_eclat_hr_at_5": 0.0, "non_trivial_baseline_hr_at_5": 0.0,
            "lift_over_baseline_pct": 0.0,
        }

    # 80/20 train/test split
    shuffled = valid_transactions.copy()
    random.shuffle(shuffled)
    split_idx = int(0.80 * n_total)
    train_txs = shuffled[:split_idx]
    test_txs  = shuffled[split_idx:]

    minsup  = EVAL_DEFAULT_MIN_SUPPORT
    minconf = EVAL_DEFAULT_MIN_CONFIDENCE
    minlift = EVAL_DEFAULT_MIN_LIFT

    # Train ECLAT strictly on training set
    miner = ECLATMiner(
        train_txs,
        min_support=minsup,
        min_absolute_count=EVAL_MIN_ABSOLUTE_COUNT,
        max_itemset_size=4,
    )
    miner.mine()
    miner.generate_rules(min_confidence=minconf, min_lift=minlift)

    # Marginal skill frequency for baseline & triviality filter
    train_counts: Dict[str, int] = {}
    for tx in train_txs:
        for s in tx:
            train_counts[s] = train_counts.get(s, 0) + 1
    n_train = len(train_txs)
    ranked_baseline_skills = sorted(train_counts.keys(), key=lambda s: -train_counts[s])

    # Leave-One-Out evaluation
    eclat_hits_at_3 = eclat_hits_at_5 = baseline_hits_at_3 = baseline_hits_at_5 = 0
    non_trivial_total = non_trivial_eclat_hits_5 = non_trivial_baseline_hits_5 = 0

    for tx in test_txs:
        masked_skill = random.choice(tx)
        input_skills = [s for s in tx if s != masked_skill]

        # ECLAT recommendations
        eclat_res = miner.recommend(input_skills, top_k=5)
        eclat_rec_skills = [r["skill"] for r in eclat_res["recommendations"]]

        # Naive baseline — most popular skills not in input
        baseline_candidates = [s for s in ranked_baseline_skills if s not in input_skills]

        if masked_skill in eclat_rec_skills[:3]:  eclat_hits_at_3 += 1
        if masked_skill in eclat_rec_skills[:5]:  eclat_hits_at_5 += 1
        if masked_skill in baseline_candidates[:3]: baseline_hits_at_3 += 1
        if masked_skill in baseline_candidates[:5]: baseline_hits_at_5 += 1

        # Non-trivial stratification
        marginal_freq = train_counts.get(masked_skill, 0) / n_train
        if marginal_freq < 0.30:
            non_trivial_total += 1
            if masked_skill in eclat_rec_skills[:5]:    non_trivial_eclat_hits_5 += 1
            if masked_skill in baseline_candidates[:5]: non_trivial_baseline_hits_5 += 1

    n_test = len(test_txs)
    eclat_hr_3  = round(eclat_hits_at_3  / n_test, 4) if n_test > 0 else 0.0
    eclat_hr_5  = round(eclat_hits_at_5  / n_test, 4) if n_test > 0 else 0.0
    base_hr_3   = round(baseline_hits_at_3 / n_test, 4) if n_test > 0 else 0.0
    base_hr_5   = round(baseline_hits_at_5 / n_test, 4) if n_test > 0 else 0.0
    nt_eclat_5  = round(non_trivial_eclat_hits_5    / non_trivial_total, 4) if non_trivial_total > 0 else 0.0
    nt_base_5   = round(non_trivial_baseline_hits_5 / non_trivial_total, 4) if non_trivial_total > 0 else 0.0
    lift_pct    = round(((eclat_hr_5 - base_hr_5) / base_hr_5) * 100, 1) if base_hr_5 > 0 else 0.0

    return {
        "role": role, "experience_mode": mode,
        "test_transactions_count": n_test,
        "eclat_hr_at_3": eclat_hr_3, "baseline_hr_at_3": base_hr_3,
        "eclat_hr_at_5": eclat_hr_5, "baseline_hr_at_5": base_hr_5,
        "non_trivial_eclat_hr_at_5": nt_eclat_5,
        "non_trivial_baseline_hr_at_5": nt_base_5,
        "lift_over_baseline_pct": lift_pct,
    }


def run_evaluation(transactions_dict: Dict) -> List[Dict]:
    """Run Leave-One-Out evaluation across all roles and modes."""
    results = []
    print("\n=== Masked Skill Recovery — Leave-One-Out Evaluation ===\n")
    for role, modes in transactions_dict.items():
        for mode in ["entry", "all"]:
            records = modes.get(mode, [])
            res = evaluate_role_mode(records, role, mode)
            results.append(res)
            if res["test_transactions_count"] > 0:
                print(f"[{role} — {mode.upper()}]  Test size: {res['test_transactions_count']}")
                print(f"  HR@3 : ECLAT {res['eclat_hr_at_3']*100:.1f}%  vs  Baseline {res['baseline_hr_at_3']*100:.1f}%")
                print(f"  HR@5 : ECLAT {res['eclat_hr_at_5']*100:.1f}%  vs  Baseline {res['baseline_hr_at_5']*100:.1f}%  (Lift: +{res['lift_over_baseline_pct']}%)")
                print(f"  Non-Trivial HR@5 : ECLAT {res['non_trivial_eclat_hr_at_5']*100:.1f}%  vs  Baseline {res['non_trivial_baseline_hr_at_5']*100:.1f}%\n")
    return results


print("✅ evaluate_role_mode() and run_evaluation() defined")

In [ ]:
# ============================================================
# Run evaluation on the synthetic demo data
# ============================================================

eval_results = run_evaluation(DEMO_TRANSACTIONS)
print("\nℹ️  Small synthetic dataset → most segments have < 20 transactions.")
print("   For full results, re-run after uploading linkedin_job_postings.csv in Section 4.")

## 📈 Section 7: Results Visualization <a id="section-7"></a>

Three matplotlib charts:
1. **Frequent Itemsets** — top 15 by count (Backend Developer, all levels)
2. **Association Rules Scatter** — x=Confidence, y=Lift, size=Support×500, coloured by consequent category
3. **Evaluation Comparison** — ECLAT vs Baseline HR@3 and HR@5 (skipped if all segments are too small)

In [ ]:
# ============================================================
# Chart 1: Frequent Itemsets — Top 15 by count
# ============================================================

sorted_itemsets_all = sorted(frequent_itemsets_demo.items(), key=lambda x: -x[1])[:15]
labels  = [", ".join(sorted(k)) for k, _ in sorted_itemsets_all]
counts  = [v for _, v in sorted_itemsets_all]
supports = [v / len(be_transactions) for v in counts]

# Colour by itemset size
size_colours = {1: "#4CAF50", 2: "#2196F3", 3: "#FF9800", 4: "#9C27B0"}
bar_colours  = [size_colours.get(len(k), "#607D8B") for k, _ in sorted_itemsets_all]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(range(len(labels)), counts, color=bar_colours, edgecolor="white", linewidth=0.5)

# Support % annotations
for i, (bar, sup) in enumerate(zip(bars, supports)):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f"{sup*100:.0f}%", va="center", fontsize=8, color="#555")

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Transaction Count", fontsize=11)
ax.set_title("Top 15 Frequent Itemsets — Backend Developer (All Levels)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, max(counts) * 1.15)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.3)

# Legend
legend_patches = [mpatches.Patch(color=c, label=f"{k}-itemset") for k, c in size_colours.items() if k <= 3]
ax.legend(handles=legend_patches, loc="lower right", fontsize=9, framealpha=0.8)

plt.tight_layout()
plt.show()
print(f"Chart 1 complete — {len(sorted_itemsets_all)} itemsets displayed")

In [ ]:
# ============================================================
# Chart 2: Association Rules — Confidence vs Lift scatter
# ============================================================

if rules_demo:
    cat_colours = {
        CAT_LANGUAGE:     "#E91E63",
        CAT_FRAMEWORK:    "#2196F3",
        CAT_DATABASE:     "#FF9800",
        CAT_CLOUD_DEVOPS: "#4CAF50",
        CAT_DEV_TOOLS:    "#9C27B0",
        "General Technology": "#607D8B",
    }

    confs    = [r["confidence"]        for r in rules_demo]
    lifts    = [r["lift"]              for r in rules_demo]
    sizes    = [r["support"] * 500     for r in rules_demo]
    colours  = [cat_colours.get(r["category"], "#607D8B") for r in rules_demo]
    labels_r = [r["consequent"][0]     for r in rules_demo]

    fig, ax = plt.subplots(figsize=(11, 7))
    sc = ax.scatter(confs, lifts, s=sizes, c=colours, alpha=0.72, edgecolors="white", linewidths=0.5)

    # Annotate top 8 rules by lift*confidence
    top_idx = sorted(range(len(rules_demo)), key=lambda i: -(rules_demo[i]["lift"] * rules_demo[i]["confidence"]))[:8]
    for i in top_idx:
        ax.annotate(labels_r[i], (confs[i], lifts[i]),
                    textcoords="offset points", xytext=(6, 4),
                    fontsize=7.5, color="#333", fontweight="bold")

    ax.axhline(y=1.0, color="red", linestyle="--", alpha=0.5, linewidth=1, label="Lift = 1.0 (random)")
    ax.axvline(x=0.5, color="gray", linestyle=":", alpha=0.4, linewidth=1)
    ax.set_xlabel("Confidence — P(B|A)", fontsize=11)
    ax.set_ylabel("Lift — Confidence / P(B)", fontsize=11)
    ax.set_title("Association Rules: Confidence vs Lift\n(Bubble size = Support × 500)", fontsize=13, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

    legend_patches = [mpatches.Patch(color=c, label=cat.split(" & ")[0]) for cat, c in cat_colours.items() if cat != "General Technology"]
    legend_patches.append(mpatches.Patch(color="gray", label="Random (Lift=1)"))
    ax.legend(handles=legend_patches, loc="upper left", fontsize=9, framealpha=0.85)

    plt.tight_layout()
    plt.show()
    print(f"Chart 2 complete — {len(rules_demo)} rules plotted")
else:
    print("No rules to plot — lower min_confidence or min_lift in Section 3.")

In [ ]:
# ============================================================
# Chart 3: ECLAT vs Baseline HR@3 and HR@5
# ============================================================

# Filter to segments that actually ran
valid_results = [r for r in eval_results if r["test_transactions_count"] > 0]

if valid_results:
    seg_labels = [f"{r['role'].split()[0]}\n({r['experience_mode']})" for r in valid_results]
    eclat_hr3   = [r["eclat_hr_at_3"]     * 100 for r in valid_results]
    base_hr3    = [r["baseline_hr_at_3"]   * 100 for r in valid_results]
    eclat_hr5   = [r["eclat_hr_at_5"]     * 100 for r in valid_results]
    base_hr5    = [r["baseline_hr_at_5"]   * 100 for r in valid_results]

    x = range(len(valid_results))
    width = 0.2

    fig, ax = plt.subplots(figsize=(max(10, len(valid_results) * 2.2), 6))
    bars1 = ax.bar([i - 1.5*width for i in x], eclat_hr3,  width, label="ECLAT HR@3",    color="#2196F3", alpha=0.85)
    bars2 = ax.bar([i - 0.5*width for i in x], base_hr3,   width, label="Baseline HR@3", color="#90CAF9", alpha=0.85)
    bars3 = ax.bar([i + 0.5*width for i in x], eclat_hr5,  width, label="ECLAT HR@5",    color="#E91E63", alpha=0.85)
    bars4 = ax.bar([i + 1.5*width for i in x], base_hr5,   width, label="Baseline HR@5", color="#F48FB1", alpha=0.85)

    for bars in [bars1, bars2, bars3, bars4]:
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                        f"{h:.1f}%", ha="center", va="bottom", fontsize=7.5)

    ax.set_xticks(list(x))
    ax.set_xticklabels(seg_labels, fontsize=9)
    ax.set_ylabel("Hit-Rate (%)", fontsize=11)
    ax.set_title("Leave-One-Out Evaluation: ECLAT vs Naive Baseline", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9, loc="upper right")
    ax.set_ylim(0, min(110, max(eclat_hr5 + base_hr5 + [10]) * 1.2))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f"Chart 3 complete — {len(valid_results)} segment(s) with sufficient data")
else:
    print("ℹ️  No segments had enough transactions for evaluation.")
    print("   Upload the full linkedin_job_postings.csv in Section 4 and re-run Sections 6–7.")

## 🌐 Section 8: Connect to the Live FastAPI Server (Optional) <a id="section-8"></a>

The production API is deployed on [Render](https://render.com) (free tier).
It may be in sleep mode — the first request wakes it (allow ~30 s).

**Base URL:** `https://eclatt-skill-gap-analyzer.onrender.com`

In [ ]:
# ============================================================
# Cell 8.1 — Health check
# ============================================================
import urllib.request

BASE_URL = "https://eclatt-skill-gap-analyzer.onrender.com"

try:
    with urllib.request.urlopen(f"{BASE_URL}/health", timeout=35) as resp:
        body = resp.read().decode()
        print(f"✅ /health  →  {resp.status}  {body}")
except Exception as e:
    print(f"⚠️  Server may be sleeping or unreachable: {e}")
    print("   Wait ~30 seconds and retry, or check https://dashboard.render.com")

In [ ]:
# ============================================================
# Cell 8.2 — List available roles
# ============================================================

try:
    with urllib.request.urlopen(f"{BASE_URL}/api/v1/roles", timeout=35) as resp:
        roles_data = json.loads(resp.read().decode())
        print(f"✅ /api/v1/roles  →  {resp.status}")
        print(json.dumps(roles_data, indent=2))
except Exception as e:
    print(f"⚠️  Could not fetch roles: {e}")

In [ ]:
# ============================================================
# Cell 8.3 — Run a live skill gap analysis
# ============================================================
import urllib.parse

payload = json.dumps({
    "role": "Backend Developer",
    "possessed_skills": ["Python", "SQL"],
    "experience_mode": "entry",
    "top_k": 5
}).encode("utf-8")

req = urllib.request.Request(
    f"{BASE_URL}/api/v1/analyze",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)

try:
    with urllib.request.urlopen(req, timeout=35) as resp:
        result = json.loads(resp.read().decode())
        print(f"✅ /api/v1/analyze  →  {resp.status}")
        print(f"\nTier used : {result.get('tier_used', 'N/A')}")
        print(f"\nRecommendations:")
        for rec in result.get("recommendations", []):
            print(f"  • {rec['skill']:<22} [{rec['category']}]  Conf={rec['confidence']:.2f}  Lift={rec['lift']:.2f}")
            print(f"    {rec['explanation']}")
except Exception as e:
    print(f"⚠️  Analyze request failed: {e}")